<img src="../assets/ga-logo.png" style="float: left; margin: 20px; height: 55px">

# APIs Minilab Crimes - Solution

---

# Scenario
You want to find out 2 things:
1. Which crimes are most common around Piccadilly Circus (in London, UK)?
2. If your bike is stolen in Oxford, how likely are you to get it back?

First of all though you'll go through a guided walk through to show you how to transform data from APIs into a `pandas` `DataFrame`.

In [ ]:
import pandas as pd
import requests

---

# Getting and analysing police data
    
Let's apply our `pandas` skills to our police data problem!
    
Let's start off by making a police API request again from this API: https://data.police.uk/docs/

In [ ]:
lat= str(51.508624722468454)
lon= str(-0.14708795945249462)
date='2023-10'

police_url = 'https://data.police.uk/api/crimes-street/all-crime?lat='+lat+'&lng='+lon+'&date='+date
police_json = requests.get(police_url).json()

We can easily convert JSON data straight into a `DataFrame` using the `pd.DataFrame()` command

In [ ]:
crimes = pd.DataFrame(police_json)

What do you notice about the DataFrame when we preview it? Something doesn't look quite right...

In [ ]:
crimes.head()

It looks like the `location` column hasn't been expanded out correctly, because the JSON in that field has a lot of nesting.

We can fix this issue using the `json_normalize` function. This expands the column out fully.

In [ ]:
pd.json_normalize(crimes['location'])

And we can join this together with our original DataFrame using the `concat` method.

In [ ]:
crimes = pd.concat([
    crimes,
    pd.json_normalize(crimes['location'])
    ],
    axis=1)

After concatenating, we can see the `location` column has been expanded out into latitude, longitude, street ID and street name.

In [ ]:
crimes.head()

As a final cleaning step, let's drop the original `location` column.

In [ ]:
crimes.drop(columns=['location'],inplace=True)

---
## Exercise: Which crimes are most common?
    
Start off by performing some extra data cleaning steps:
    
* Remove the '-' in the `category` column
* Remove the phrase "on or near" in the `street.name` column

In [ ]:
crimes['category'] = crimes['category'].str.replace('-',' ')
crimes['street.name'] = crimes['street.name'].str.replace('On or near ','')

Which streets were the most crimes recorded on?

In [ ]:
crimes['street.name'].value_counts()

What kinds of crimes are the most common?

In [ ]:
crimes['category'].value_counts()

---
## Exercise: How many crimes are solved? </font>
    
Now, let's combine our API and `pandas` skills to tackle a question.
    
We're now interested in how many crimes are **solved** in different parts of the UK. Can you use the **street level outcomes** section of the Police API (https://data.police.uk/docs/method/outcomes-at-location/) + your `pandas` skills to answer the question:
    
**If your bike is stolen in Oxford, how likely are you to get it back?**  (Either in Oxford itself, or for bonus points work out the surrounding area.)
    
There are several potential ways to answer this question - think about how you'll answer it before starting to code.
    
Remember to read the API documentation carefully, and experiment with demo API requests in your web browser **before** starting to write any code!

---
## <font color='green'> Sample solution: How many crimes are solved?
    
There are a few possible ways to answer this question.  In this sample solution we'll do it in 3 steps:
    
1. Determine how we can calculate the likelihood (to do this we'll need to understand what data is provided in the endpoint)
2. Get the data for one month and calculate the likelihood
3. Get all the data we can and work out the likelihood based on that

## 1. Determine how we can calculate the likelihood

The likelihood - or probability - of getting a bike back that has been stolen in Oxford can be calculated as:

$$\frac{The\; number\; of\; times\; bikes\; were\; returned}{Total\; number\; of\; bike\; thefts}$$

in any given time period.

By exploring the **Street-level outcomes** endpoint https://data.police.uk/docs/method/outcomes-at-location/ we can see that there are 28 categories of outcome for a crime:

- Awaiting court outcome
- Court result unavailable
- Court case unable to proceed
- Local resolution
- etc


Interestingly, there are no categories for property being returned to a victim, so there is no absolute way to tell from this data how many bikes are returned to people who have had them stolen.

However, we can make an assumption.

For the purposes of this exercise, we will assume that 'Local resolution' means that the bike was returned to the owner, and that no further police action was taken.  So our likelihood of getting a bike returned is now:

$$\frac{Number\; of\; Local\; Resolutions}{Total\; number\; of\; bike\; thefts}$$

We can get the **location of Oxford** by using Google Maps. For now we will just use one point (rather than multiple points, or the `poly` parameter) as the endpoint provides data within 1 mile radius of the point.


    latitude: 51.75157809244022
    longitude: -1.2565147281330131

## 2. Get the data for one month and calculate the likelihood

We can get the data for one month and calculate the likelihood using the `requests` library.

Let's start by getting all outcomes back for a given month.

In [ ]:
lat= str(51.75157809244022)
lon= str(-1.2565147281330131)
date='2024-02'

street_url = 'https://data.police.uk/api/outcomes-at-location?date='+date+'&lat='+lat+'&lng='+lon

street_json = requests.get(street_url)

data = street_json.json()
# data

You get probably get hundreds of items back for the month you have selected

In [ ]:
len(street_json.json())

The JSON is a list of dictionaries, with each dictionary representing an individual outcome of a crime.

Let's inspect the make up of one outcome to see how it is structured:

In [ ]:
data[0]

There are 2 categories for each outcome: the category of the outcome e.g. `Unable to prosecute`; and the category of the crime e.g. `violent-crime`.

To find **bike thefts**, we need to look at the categories of crimes: https://data.police.uk/docs/method/outcomes-at-location/

(You can link to this page from the Street-level outcomes page)

In [ ]:
requests.get('https://data.police.uk/api/crime-categories?date=2023-10').json()

So what we need to do to get **bike theft** data is to extract only those outcomes which have a crime category = `'bicycle-theft'`, and then count up how many of each category of the outcome we have.

We can use a `for` loop to do this.

In [ ]:
data[0]['category']['code']

In [ ]:
# create a dictionary to put the bike theft data into
bike_theft_data = {}

# iterate over each outcome, and if the crime category is 'bicycle-theft', then add the outcome category to the
# dictionary as a key, and give it a value of 1.  If the category already exists in the dictionary, then increment
# its value by 1.

for outcome in data:
    # print("for loop:", outcome)
    if outcome['crime']['category'] == 'bicycle-theft':
        if outcome['category']['code'] not in bike_theft_data:
            bike_theft_data[outcome['category']['code']] = 1
            # print(bike_theft_data)
            print("forloop_if:",bike_theft_data)
        else:
            bike_theft_data[outcome['category']['code']] += 1
            print(".....")
            print("forloop_else:",bike_theft_data)

bike_theft_data

We now have the data we need to be able to calculate the likelihood given the data for one month.

In [ ]:
# get the total of all outcomes relating to bike theft:

sum(bike_theft_data.values())

Since there are **no** Local Resolutions, the likelihood of getting your bike back given this data is 0.

But drawing that conclusion from only one month of data is not very sound as it may not be that representative of all months.  Let's get more data.

## 3. Get all the data we can and work out the likelihood based on that

In order to get data for many months, we can run the same code that we have already used, but use another `for` loop to get the data for each month.

First, we need to list out the months that we want to get data for.

The endpoint only provides the most recent 3 years of data.  Let's get all of it.

In [ ]:
# create a list of 3 years of dates to gather outcomes for

dates = []
years = [2022, 2023, 2024, 2025]

for year in years:
    for month in range(1,13):
        dates.append(str(year)+'-'+str(month))

dates

Let's create a list to contain all the data we get from the API.  (We will then use this to create a pandas DataFrame.)

In [ ]:
all_data = []

Now we can tweak the code we used before (to get one month of data), so we can capture the month that we are getting data for, and we can get all the data for each month.

In [ ]:
lat= str(51.75157809244022)
lon= str(-1.2565147281330131)


for date in dates:
    bike_theft_data = {}
    # add the month to the dictionary
    bike_theft_data['month'] = date

    # get bike theft data for each month
    street_url = 'https://data.police.uk/api/outcomes-at-location?date='+date+'&lat='+lat+'&lng='+lon
    street_json = requests.get(street_url)

    # some of the months will not be available (either they are more then 3 years ago, or have not happened yet)
    # these 2 lines make the code move to the next iteration if no data is returned, and allows the code to keep
    # running without error
    if street_json.status_code == 404:
        continue
    data = street_json.json()

    # use the same for loop as before to extract the bike theft outcome categories
    for outcome in data:
        if outcome['crime']['category'] == 'bicycle-theft':
            if outcome['category']['code'] not in bike_theft_data:
                bike_theft_data[outcome['category']['code']] = 1
            else:
                bike_theft_data[outcome['category']['code']] += 1
    # append the data for each month to the list of all data
    all_data.append(bike_theft_data)

In [ ]:
all_data

We can turn the data we have extracted into a pandas DataFrame.

In [ ]:
df = pd.DataFrame(all_data)
df

We can see that there is now a **Local Resolution** category in some months.

To work out the likelihood of getting your bike back, we need to sum up all the **local-resolution**s, and sum up all the data.

In [ ]:
# Add a 'Total' row that sums up each column
df.loc['Total'] = df.sum()

# Replace the value in the Month column
df.loc['Total', 'month'] = 'All'

df

In [ ]:
# get the total number of bike thefts that have had an outcome confirmed during the time period

total_bike_thefts = df.loc['Total'][1:].sum()
total_bike_thefts

In [ ]:
# Work out the likelihood

likelihood = df['local-resolution'].sum() / total_bike_thefts
print(f'The likelihood of getting your bike back is about {likelihood*100: .2f}%')

# Bonus

We can broaden our search to be more exact about the space we are seaching in.  To do this we use the `poly` parameter specified in the documentation.

In [ ]:
all_data = []

In [ ]:
# points around Oxford to create a broader area to search within

a = '51.787796997740735, -1.2837637088905183'
b = '51.79520329585142, -1.2715083485059857'
c = '51.771137196203334, -1.202505031426324'
d = '51.764523199909554, -1.177860757691506'
e = '51.72302745869018, -1.1829530687349075'
f = '51.711069927298624, -1.1979785311564242'
g = '51.70861571067208, -1.2211768387677524'
h = '51.717263310493465, -1.2351335441599336'
i = '51.72974878764573, -1.2786994807013587'
j = '51.75247029588256, -1.2812766516804153'

poly=a+':'+b+':'+c+':'+d+':'+e+':'+f+':'+g+':'+h+':'+i+':'+j
poly

In [ ]:
for date in dates:
    bike_theft_data = {}
    # add the month to the dictionary
    bike_theft_data['month'] = date

    # get bike theft data for each month
    street_url = 'https://data.police.uk/api/outcomes-at-location?date='+date+'&poly='+poly
    street_json = requests.get(street_url)
    
    
    # as in the previous exercise, some of the months will not be available (either they are more then 3 years ago, or have not happened yet)
    # these 2 lines make the code move to the next iteration if no data is returned, and allows the code to keep
    # running without error
    if street_json.status_code == 404:
        continue
    data = street_json.json()

    # use the same for loop as before to extract the bike theft outcome categories
    for outcome in data:
        if outcome['crime']['category'] == 'bicycle-theft':
            if outcome['category']['code'] not in bike_theft_data:
                bike_theft_data[outcome['category']['code']] = 1
            else:
                bike_theft_data[outcome['category']['code']] += 1
    # append the data for each month to the list of all data
    all_data.append(bike_theft_data)

In [ ]:
df = pd.DataFrame(all_data)
df

In [ ]:
# Add a 'Total' row that sums up each column
df.loc['Total'] = df.sum()

# Replace the value in the Month column
df.loc['Total', 'month'] = 'All'

df

In [ ]:
total_bike_thefts = df.loc['Total'][1:].sum()
likelihood = df['local-resolution'].sum() / total_bike_thefts
print(f'The likelihood of getting your bike back is about {likelihood*100: .2f}%')

## Tip
- Learn about [Counter](https://realpython.com/python-counter/), the pythonic way to count objects (items)

---